# Practical No. 06 — Retrieval-Augmented Generation (RAG) with a Pre-trained LLM

**Student:** Mit Jangid  
**PRN:** 202401110072  
**Batch:** A1  
**Topic:** Build a document-grounded question-answering system and compare it with an LLM without retrieval.

> This is an independently structured implementation based on the assignment requirements. It uses a different fictional institute, a different document collection, different chunking settings, a different retrieval configuration, and a separate evaluation workflow from the reference report.

## Learning outcomes
- Load and normalize a small collection of text documents.
- Split documents into overlapping chunks and preserve source metadata.
- Create semantic embeddings and store them in ChromaDB.
- Retrieve relevant passages for a user question.
- Compare a pretrained LLM's response with and without retrieved context.
- Evaluate factual correctness and retrieval coverage on a small test set.

## System overview

`Handbook files → Text normalization → Overlapping chunks → Embeddings → ChromaDB → Top-k retrieval → Context-aware prompt → Answer with sources`

## 1. Set up the Colab environment

Run this cell once. A GPU runtime is recommended for loading the language model. In Colab, select **Runtime → Change runtime type → T4 GPU** if available.

In [1]:

!pip -q install transformers accelerate sentence-transformers chromadb pypdf


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
t

## 2. Imports and runtime configuration

We use a compact instruction-tuned model so the experiment can run on a Colab GPU with limited memory. The embedding model is separate from the answer-generation model: embeddings are used for search, while the LLM writes the final response.

In [2]:
import os
import re
import time
import torch
import pandas as pd
import chromadb

from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Runtime device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Runtime device: cuda
GPU: Tesla T4


## 3. Create a small domain-specific knowledge base

Instead of reusing the university and documents from the reference assignment, this notebook uses a fictional **Greenfield Institute of Technology (GIT)** and a fresh set of student-support documents. The information is intentionally fictional and is only used to test retrieval and grounding.

In [3]:
documents = {
    "academic_support.txt": '''
GREENFIELD INSTITUTE — ACADEMIC SUPPORT
Students must maintain at least 75 percent attendance in each registered course to be eligible for the end-semester examination. A student may apply for an attendance review within five working days after the attendance statement is published.
The project proposal must be submitted to the department office by 4:00 PM on the announced deadline. Late project proposals require written approval from the project coordinator.
Students can request one re-evaluation per subject within seven calendar days after results are declared. The re-evaluation application charge is Rs. 250 per subject.
''',
    "campus_services.txt": '''
GREENFIELD INSTITUTE — CAMPUS SERVICES
The central library is open from 8:30 AM to 8:00 PM on weekdays and from 9:00 AM to 1:00 PM on Saturdays. Students may borrow up to three books at a time for a period of 12 days.
A library book returned after its due date attracts a fine of Rs. 5 for each overdue day. The library help desk is located on the ground floor of the learning resource centre.
The computer centre operates from 9:00 AM to 6:00 PM, Monday to Friday. Students must use their institute login to access the lab computers.
''',
    "residence_and_wellbeing.txt": '''
GREENFIELD INSTITUTE — RESIDENCE AND WELLBEING
The residence hall entry gate closes at 10:00 PM. A student returning after the gate closing time must record the reason in the late-entry register and inform the residence supervisor.
Visitors are permitted in the common lounge between 4:00 PM and 7:00 PM. Visitors are not allowed inside student rooms.
The Student Support Desk is coordinated by Dr. Kavita Rao. Students can submit a confidential wellbeing request through the support desk during working hours, 10:00 AM to 5:00 PM.
'''
}

DATA_DIR = Path("/content/git_handbook")
DATA_DIR.mkdir(exist_ok=True)

for filename, content in documents.items():
    (DATA_DIR / filename).write_text(content.strip(), encoding="utf-8")

print("Created files:")
for path in sorted(DATA_DIR.glob("*.txt")):
    print("-", path.name, "|", len(path.read_text(encoding="utf-8")), "characters")

Created files:
- academic_support.txt | 630 characters
- campus_services.txt | 534 characters
- residence_and_wellbeing.txt | 531 characters


## 4. Load and normalize the documents

The loader reads all `.txt` files in the knowledge-base folder. Normalization removes carriage returns, trims extra spaces, and reduces repeated blank lines. The original filename is retained as metadata so the final answer can identify its source.

In [4]:
def normalize_text(text):
    text = text.replace("\\r", "")
    text = re.sub(r"[ \\t]+", " ", text)
    text = re.sub(r"\\n{3,}", "\\n\\n", text)
    return text.strip()

def load_text_folder(folder):
    records = []
    for file_path in sorted(Path(folder).glob("*.txt")):
        raw_text = file_path.read_text(encoding="utf-8")
        cleaned = normalize_text(raw_text)
        records.append({
            "source": file_path.name,
            "text": cleaned
        })
    return records

loaded_docs = load_text_folder(DATA_DIR)
print("Documents loaded:", len(loaded_docs))
for doc in loaded_docs:
    print(f"{doc['source']}: {len(doc['text'])} characters")

Documents loaded: 3
academic_support.txt: 605 characters
campus_services.txt: 519 characters
residence_and_wellbeing.txt: 511 characters


## 5. Split documents into overlapping chunks

This implementation uses a character-window splitter with **350 characters per chunk** and **60 characters of overlap**. Each chunk stores its source filename and chunk index. Overlap helps retain facts that might otherwise be split at a boundary.

In [5]:
CHUNK_SIZE = 350
CHUNK_OVERLAP = 60

def split_into_chunks(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    if chunk_size <= overlap:
        raise ValueError("chunk_size must be greater than overlap")

    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))

        # Prefer a nearby paragraph or sentence boundary when possible.
        if end < len(text):
            boundary = max(text.rfind("\\n", start, end),
                           text.rfind(". ", start, end))
            if boundary > start + int(chunk_size * 0.55):
                end = boundary + 1

        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)

        if end >= len(text):
            break
        start = max(end - overlap, start + 1)

    return chunks

chunk_records = []
for doc in loaded_docs:
    pieces = split_into_chunks(doc["text"])
    for idx, piece in enumerate(pieces):
        chunk_records.append({
            "id": f"{doc['source']}__part_{idx}",
            "source": doc["source"],
            "chunk_index": idx,
            "text": piece
        })

print("Total chunks:", len(chunk_records))
display(pd.DataFrame([{
    "id": x["id"], "source": x["source"],
    "chunk_index": x["chunk_index"], "characters": len(x["text"])
} for x in chunk_records]))

Total chunks: 6


,id,source,chunk_index,characters
0,academic_support.txt__part_0,academic_support.txt,0,350
1,academic_support.txt__part_1,academic_support.txt,1,315
2,campus_services.txt__part_0,campus_services.txt,0,300
3,campus_services.txt__part_1,campus_services.txt,1,279
4,residence_and_wellbeing.txt__part_0,residence_and_wellbeing.txt,0,294
5,residence_and_wellbeing.txt__part_1,residence_and_wellbeing.txt,1,277


## 6. Generate embeddings and create the ChromaDB collection

The embedding model converts each chunk into a dense numerical representation. ChromaDB stores the text, vector representation, and metadata. We use cosine distance for semantic similarity search.

In [6]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)

chunk_texts = [item["text"] for item in chunk_records]
chunk_vectors = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
).tolist()

# Re-create the collection so that rerunning the notebook does not duplicate records.
client = chromadb.Client()
try:
    client.delete_collection("greenfield_handbook")
except Exception:
    pass

collection = client.create_collection(
    name="greenfield_handbook",
    metadata={"hnsw:space": "cosine"}
)

collection.add(
    ids=[item["id"] for item in chunk_records],
    documents=chunk_texts,
    embeddings=chunk_vectors,
    metadatas=[{
        "source": item["source"],
        "chunk_index": item["chunk_index"]
    } for item in chunk_records]
)

print("Vectors stored:", collection.count())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Vectors stored: 6


## 7. Retrieve relevant passages

A question is embedded with the same model and sent to ChromaDB. The function returns the top four matching chunks, along with their source names and similarity scores. Here, similarity is calculated as `1 - cosine_distance`.

In [7]:
TOP_K = 4

def retrieve_passages(question, top_k=TOP_K):
    query_vector = embedding_model.encode(
        [question], normalize_embeddings=True
    )[0].tolist()

    result = collection.query(
        query_embeddings=[query_vector],
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    matches = []
    for text, metadata, distance in zip(
        result["documents"][0],
        result["metadatas"][0],
        result["distances"][0]
    ):
        matches.append({
            "text": text,
            "source": metadata["source"],
            "chunk_index": metadata["chunk_index"],
            "similarity": round(1 - float(distance), 3)
        })
    return matches

sample_question = "How much is charged for each day a library book is overdue?"
sample_matches = retrieve_passages(sample_question)

display(pd.DataFrame(sample_matches))

,text,source,chunk_index,similarity
0,r i s due da e a rac s a fine of Rs. 5 for eac...,campus_services.txt,1,0.465
1,l mus be submi ed o he depar men office by 4:0...,academic_support.txt,1,0.281
2,GREENFIELD INSTITUTE — CAMPUS SERVICES\nThe ce...,campus_services.txt,0,0.223
3,GREENFIELD INSTITUTE — ACADEMIC SUPPORT\nS ude...,academic_support.txt,0,0.125


## 8. Load the pretrained instruction model

This notebook uses `Qwen/Qwen2.5-1.5B-Instruct`. It is an open instruction-tuned model. The model is loaded in half precision on GPU when CUDA is available; otherwise it uses full precision on CPU, which may be slow.

In [8]:
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
model_kwargs = {"device_map": "auto"}
if torch.cuda.is_available():
    model_kwargs["torch_dtype"] = torch.float16

llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    **model_kwargs
)
llm.eval()

print("Loaded:", LLM_NAME)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-1.5B-Instruct


## 9. Define generation functions: LLM-only and RAG

The two functions use the same language model and decoding settings. The difference is that the RAG function adds retrieved passages and source labels to the prompt. The RAG instruction explicitly tells the model not to invent information when the documents do not contain an answer.

In [9]:
def generate_response(system_instruction, user_message, max_new_tokens=160):
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_message}
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(llm.device)

    with torch.no_grad():
        output = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def answer_without_retrieval(question):
    instruction = (
        "You are a helpful assistant. Answer the user's question briefly. "
        "If you are uncertain, say that you are not sure."
    )
    return generate_response(instruction, question)


def answer_with_rag(question, top_k=TOP_K):
    passages = retrieve_passages(question, top_k=top_k)

    context_blocks = []
    for number, item in enumerate(passages, start=1):
        context_blocks.append(
            f"[{number}] Source: {item['source']}\\n{item['text']}"
        )
    context = "\\n\\n".join(context_blocks)

    instruction = (
        "You answer questions using only the supplied context. "
        "Do not use outside knowledge or guess. If the context does not contain "
        "the answer, reply exactly: I don't know based on the provided documents. "
        "Cite supporting sources using their bracket numbers, for example [1]."
    )
    user_message = f"Context:\\n{context}\\n\\nQuestion: {question}"
    return generate_response(instruction, user_message), passages

## 10. Inspect one question in both modes

Use a question whose answer is present in the fictional documents. Review the retrieved passages and compare the LLM-only answer with the context-grounded RAG answer.

In [11]:
question = "What percentage of attendance is required to sit for end-semester exams?"

plain_answer = answer_without_retrieval(question)
rag_answer, evidence = answer_with_rag(question)

print("QUESTION:", question)
print("\\nLLM-ONLY ANSWER:\\n", plain_answer)
print("\\nRAG ANSWER:\\n", rag_answer)
print("\\nRETRIEVED EVIDENCE:")
display(pd.DataFrame(evidence))

QUESTION: What percentage of attendance is required to sit for end-semester exams?
\nLLM-ONLY ANSWER:\n The exact percentage can vary by institution, but typically it's around 75%. Students must attend at least 75% of classes or have permission from their instructor to be eligible to take end-of-term exams.
\nRAG ANSWER:\n According to the context, 75% of attendance is required to be eligible for the end-semester exam. This information is sourced from Academic Support (Source [1]).
\nRETRIEVED EVIDENCE:


,text,source,chunk_index,similarity
0,GREENFIELD INSTITUTE — ACADEMIC SUPPORT\nS ude...,academic_support.txt,0,0.292
1,r i s due da e a rac s a fine of Rs. 5 for eac...,campus_services.txt,1,0.266
2,re permi ed in he common lounge be ween 4:00 P...,residence_and_wellbeing.txt,1,0.242
3,l mus be submi ed o he depar men office by 4:0...,academic_support.txt,1,0.179


## 11. Build a small evaluation set

The evaluation set contains eight questions about the fictional institute and one general-knowledge control question. The gold facts are used for a simple keyword-based comparison. This is a lightweight classroom metric, not a complete measure of answer quality.

In [12]:
test_cases = [
    {
        "question": "What attendance percentage is required for end-semester exams?",
        "gold": "75 percent",
        "keywords": ["75 percent", "75%"],
        "source": "academic_support.txt"
    },
    {
        "question": "How long can a student borrow library books?",
        "gold": "12 days",
        "keywords": ["12 days"],
        "source": "campus_services.txt"
    },
    {
        "question": "What is the fine for each overdue library day?",
        "gold": "Rs. 5 per day",
        "keywords": ["rs. 5", "5 per", "5 rupees"],
        "source": "campus_services.txt"
    },
    {
        "question": "At what time does the residence hall gate close?",
        "gold": "10:00 PM",
        "keywords": ["10:00 pm", "10 pm"],
        "source": "residence_and_wellbeing.txt"
    },
    {
        "question": "Who coordinates the Student Support Desk?",
        "gold": "Dr. Kavita Rao",
        "keywords": ["kavita rao"],
        "source": "residence_and_wellbeing.txt"
    },
    {
        "question": "How many days are allowed to request re-evaluation?",
        "gold": "seven calendar days",
        "keywords": ["seven calendar days", "7 days"],
        "source": "academic_support.txt"
    },
    {
        "question": "What is the re-evaluation application charge per subject?",
        "gold": "Rs. 250",
        "keywords": ["rs. 250", "250"],
        "source": "academic_support.txt"
    },
    {
        "question": "What are the weekday library opening hours?",
        "gold": "8:30 AM to 8:00 PM",
        "keywords": ["8:30 am", "8:00 pm"],
        "source": "campus_services.txt"
    },
    {
        "question": "What is the capital of Japan?",
        "gold": "Tokyo",
        "keywords": ["tokyo"],
        "source": "general knowledge"
    }
]
print("Evaluation questions:", len(test_cases))
display(pd.DataFrame(test_cases)[["question", "gold", "source"]])

Evaluation questions: 9


,question,gold,source
0,What attendance percentage is required for end...,75 percent,academic_support.txt
1,How long can a student borrow library books?,12 days,campus_services.txt
2,What is the fine for each overdue library day?,Rs. 5 per day,campus_services.txt
3,At what time does the residence hall gate close?,10:00 PM,residence_and_wellbeing.txt
4,Who coordinates the Student Support Desk?,Dr. Kavita Rao,residence_and_wellbeing.txt
5,How many days are allowed to request re-evalua...,seven calendar days,academic_support.txt
6,What is the re-evaluation application charge p...,Rs. 250,academic_support.txt
7,What are the weekday library opening hours?,8:30 AM to 8:00 PM,campus_services.txt
8,What is the capital of Japan?,Tokyo,general knowledge


## 12. Run the evaluation

For each question, the notebook records both answers and checks whether at least one gold keyword appears. It also checks whether the expected source document is represented in the retrieved top-k results. The final question is outside the handbook and is expected to test whether the RAG prompt can decline to answer.

In [13]:
def contains_gold(answer, keywords):
    answer_lower = answer.lower()
    return any(word.lower() in answer_lower for word in keywords)

evaluation_rows = []

for item in test_cases:
    q = item["question"]

    start = time.time()
    llm_only = answer_without_retrieval(q)
    llm_seconds = round(time.time() - start, 2)

    start = time.time()
    rag_text, retrieved = answer_with_rag(q)
    rag_seconds = round(time.time() - start, 2)

    retrieved_sources = sorted(set(x["source"] for x in retrieved))
    is_control = item["source"] == "general knowledge"

    evaluation_rows.append({
        "Question": q,
        "Gold fact": item["gold"],
        "LLM-only correct": contains_gold(llm_only, item["keywords"]),
        "RAG correct": contains_gold(rag_text, item["keywords"]),
        "Retrieval hit": (item["source"] in retrieved_sources) if not is_control else None,
        "LLM-only answer": llm_only,
        "RAG answer": rag_text,
        "Retrieved sources": ", ".join(retrieved_sources),
        "LLM seconds": llm_seconds,
        "RAG seconds": rag_seconds
    })

results_df = pd.DataFrame(evaluation_rows)
display(results_df[[
    "Question", "Gold fact", "LLM-only correct",
    "RAG correct", "Retrieval hit"
]])

,Question,Gold fact,LLM-only correct,RAG correct,Retrieval hit
0,What attendance percentage is required for end...,75 percent,False,True,True
1,How long can a student borrow library books?,12 days,False,False,True
2,What is the fine for each overdue library day?,Rs. 5 per day,False,True,True
3,At what time does the residence hall gate close?,10:00 PM,False,True,True
4,Who coordinates the Student Support Desk?,Dr. Kavita Rao,False,False,True
5,How many days are allowed to request re-evalua...,seven calendar days,False,True,True
6,What is the re-evaluation application charge p...,Rs. 250,False,True,True
7,What are the weekday library opening hours?,8:30 AM to 8:00 PM,False,True,True
8,What is the capital of Japan?,Tokyo,True,False,None


## 13. Summarize evaluation metrics

The following summary calculates the number of correct answers for both modes and the number of in-domain questions for which the expected source appeared in the retrieved set. Because the test set is small and uses keyword matching, interpret the numbers as a basic experiment rather than a general benchmark.

In [14]:
in_domain = results_df[results_df["Retrieval hit"].notna()]

summary = pd.DataFrame({
    "Metric": [
        "LLM-only factual matches",
        "RAG factual matches",
        "Retrieval hits (in-domain questions)",
        "Number of in-domain questions",
        "Total questions"
    ],
    "Value": [
        int(results_df["LLM-only correct"].sum()),
        int(results_df["RAG correct"].sum()),
        int(in_domain["Retrieval hit"].sum()),
        len(in_domain),
        len(results_df)
    ]
})
display(summary)

results_df.to_csv("rag_comparison_results.csv", index=False)
print("Saved: rag_comparison_results.csv")

,Metric,Value
0,LLM-only factual matches,1
1,RAG factual matches,6
2,Retrieval hits (in-domain questions),8
3,Number of in-domain questions,8
4,Total questions,9


Saved: rag_comparison_results.csv


## 14. Out-of-scope question

Ask for information that is deliberately absent from all three documents. The RAG response should follow its instruction to say that the information is not available in the supplied documents. Compare it with the LLM-only response.

In [15]:
outside_question = "What is the annual tuition fee for the institute's MBA programme?"

outside_plain = answer_without_retrieval(outside_question)
outside_rag, outside_evidence = answer_with_rag(outside_question)

print("QUESTION:", outside_question)
print("\\nLLM-ONLY:\\n", outside_plain)
print("\\nRAG:\\n", outside_rag)
print("\\nRetrieved context:")
display(pd.DataFrame(outside_evidence))

QUESTION: What is the annual tuition fee for the institute's MBA programme?
\nLLM-ONLY:\n I'm sorry, but I don't have specific information about the current annual tuition fees for an MBA program at any particular institute. Tuition fees can vary significantly between institutions and even within different programs offered by the same institution. To get accurate and up-to-date information, I would recommend checking directly with the admissions office of the specific institute or university you're interested in attending. They should be able to provide you with the most current and relevant details regarding their MBA program's costs.
\nRAG:\n I don't know based on the provided documents.
\nRetrieved context:


,text,source,chunk_index,similarity
0,GREENFIELD INSTITUTE — ACADEMIC SUPPORT\nS ude...,academic_support.txt,0,0.330
1,l mus be submi ed o he depar men office by 4:0...,academic_support.txt,1,0.274
2,GREENFIELD INSTITUTE — CAMPUS SERVICES\nThe ce...,campus_services.txt,0,0.271
3,r i s due da e a rac s a fine of Rs. 5 for eac...,campus_services.txt,1,0.253


## 15. Observations and discussion

After running all cells, use the generated output to discuss these points:

1. Did semantic retrieval return a chunk from the expected document?
2. Did the RAG answer include the correct fact and a source citation?
3. Did the LLM-only answer guess, omit, or misstate any institute-specific fact?
4. Did the RAG system refuse to invent the MBA tuition fee?
5. Which questions were missed by retrieval, and what could improve the results?

### Limitations
- The knowledge base contains only three short text files.
- Character-based chunking can split a fact across chunk boundaries.
- A fixed top-k value may miss relevant passages.
- Keyword-based evaluation does not fully judge semantic correctness.
- A small language model may occasionally ignore instructions or misread retrieved context.

### Possible improvements
- Compare different chunk sizes and overlap values.
- Add PDF ingestion and metadata filtering.
- Use hybrid retrieval combining keyword and vector search.
- Add a cross-encoder reranker.
- Evaluate faithfulness and answer relevance with human review.

## 16. Conclusion

A document-grounded RAG question-answering system was implemented using text normalization, overlapping chunk creation, MiniLM embeddings, ChromaDB semantic retrieval, and a pretrained Qwen instruction model. The experiment compares answers generated by the LLM alone with answers generated using retrieved context. The evaluation and out-of-scope test demonstrate how retrieval can make answers more specific to a private document collection and help the model decline questions that the supplied documents cannot answer.

**Submission note:** Run the notebook from top to bottom, inspect the actual generated answers and metrics, and retain the executed outputs before exporting or submitting the practical.